In [ ]:
from utils import * 
from scipy.spatial import ConvexHull, Delaunay, procrustes
from numpy.linalg import norm
from sklearn.metrics import pairwise_distances




# Trial-level

In [ ]:
#-------------------- NLP features

from typing import Dict

def replace_pronouns(df: pd.DataFrame, character_genders: Dict[str, str]) -> pd.DataFrame:
    """
    Replace bracketed placeholders in columns ['text','opt1_text','opt2_text']
    using only the provided character_genders mapping.
    """

    required_cols = {'text', 'opt1_text', 'opt2_text'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame missing required columns: {missing}")

    # ---------- helpers ----------
    def forms_for_gender(g: str):
        g = (g or "").strip().lower()
        if g == "woman":
            return dict(
                subj="she", obj="her", poss_det="her", poss_pron="hers", refl="herself",
                is_contr="she's", would_contr="she'd", will_contr="she'll",
                guy_girl="girl", sir_maam="ma'am", man_woman="woman", mr_mrs="Mrs."
            )
        # default to 'man'
        return dict(
            subj="he", obj="him", poss_det="his", poss_pron="his", refl="himself",
            is_contr="he's", would_contr="he'd", will_contr="he'll",
            guy_girl="guy", sir_maam="sir", man_woman="man", mr_mrs="Mr."
        )

    def name_for_role(role: str, gender: str) -> str:
        gender = (gender or "man").lower()
        if role in ("First", "Second"):
            return "Chris" if gender == "man" else "Jessica"
        if role in ("Assistant", "Neutral"):
            return "Anthony" if gender == "man" else "Kayce"
        if role == "Newcomb":
            return "Newcomb"
        if role == "Hayworth":
            return "Hayworth"
        # fallback
        return role

    roles = ["First", "Second", "Assistant", "Newcomb", "Hayworth", "Neutral"]
    role_forms = {r: forms_for_gender(character_genders.get(r, "man")) for r in roles}
    role_names = {r: name_for_role(r, character_genders.get(r, "man")) for r in roles}

    # Newcomb spouse (opposite gender)
    newcomb_gender = (character_genders.get("Newcomb", "man") or "man").lower()
    spouse_gender = "woman" if newcomb_gender == "man" else "man"
    spouse_forms = forms_for_gender(spouse_gender)
    spouse_name = "Mary" if spouse_gender == "woman" else "James"

    # ---------- build replacement rules ----------
    replacements = []

    def pat(token: str) -> re.Pattern:
        return re.compile(re.escape(token))

    for role in roles:
        f = role_forms[role]
        nm = role_names[role]
        mapping = {
            f"[{role} character pronoun]": f["subj"],
            f"[{role} character objective]": f["obj"],
            f"[{role} character reflexive]": f["refl"],
            f"[{role} character possessive pronoun]": f["poss_pron"],
            f"[{role} character possessive determiner]": f["poss_det"],
            f"[{role} character 'is' contraction]": f["is_contr"],
            f"[{role} character 'would' contraction]": f["would_contr"],
            f"[{role} character 'will' contraction]": f["will_contr"],
            f"[Mr./Mrs. {role} character]": f["mr_mrs"],
            f"[{role} character sir/ma'am]": f["sir_maam"],
            f"[{role} character guy/girl]": f["guy_girl"],
            f"[{role} character man/woman]": f["man_woman"],
            f"[{role} character name]": nm,
            f"[{role} character first name]": nm,
        }
        # generic possessive → pronoun possessive determiner, except Newcomb
        if role != "Newcomb":
            mapping[f"[{role} character possessive]"] = f["poss_det"]

        for k, v in mapping.items():
            replacements.append((pat(k), v))

    # Newcomb-specific overrides & spouse
    replacements.append((pat("[Newcomb character possessive]"), "Newcomb's"))
    replacements.append((pat("[Newcomb character spouse's name]"), spouse_name))
    replacements.append((pat("[Newcomb character spouse's pronoun]"), spouse_forms["subj"]))

    # Explicit tokens in your scripts
    replacements.append((pat("[Hayworth character sir/ma'am]"), role_forms["Hayworth"]["sir_maam"]))
    replacements.append((pat("[Second character guy/girl]"), role_forms["Second"]["guy_girl"]))
    replacements.append((pat("[Newcomb character man/woman]"), role_forms["Newcomb"]["man_woman"]))

    # Collapse double spaces after substitution
    space_fix = re.compile(r" {2,}")

    # ---------- build pronoun pattern for capitalization ----------
    pronoun_tokens = set()
    for forms in list(role_forms.values()) + [spouse_forms]:
        for key in ["subj", "obj", "poss_det", "poss_pron", "refl",
                    "is_contr", "would_contr", "will_contr"]:
            pronoun_tokens.add(forms[key])

    # Longer first, escaped, then joined into a single alternation group
    escaped = [re.escape(p) for p in sorted(pronoun_tokens, key=len, reverse=True)]
    pronoun_pattern = r"(?:" + "|".join(escaped) + r")"

    # Start-of-string or start-of-quote pronouns
    sent_start_re = re.compile(
        r"""^([\s"“”']*)(?P<pronoun>""" + pronoun_pattern + r""")\b"""
    )

    # Pronouns after ., !, ? with optional quotes/spaces
    sent_after_punct_re = re.compile(
        r"""([\.!\?]["“”'\s]*)(?P<pronoun>""" + pronoun_pattern + r""")\b"""
    )

    def _capitalize_sentence_initial_pronouns(text: str) -> str:
        # Start of string / quote
        text = sent_start_re.sub(
            lambda m: m.group(1) + m.group("pronoun").capitalize(), text
        )
        # After punctuation
        text = sent_after_punct_re.sub(
            lambda m: m.group(1) + m.group("pronoun").capitalize(), text
        )
        return text

    def _apply(text):
        if not isinstance(text, str):
            return text
        out = text
        # bracket replacements
        for rx, repl in replacements:
            out = rx.sub(repl, out)
        # cleanup spaces
        out = space_fix.sub(" ", out)
        # capitalization at sentence / quote boundaries
        out = _capitalize_sentence_initial_pronouns(out)
        return out

    out = df.copy()
    for col in ["text", "opt1_text", "opt2_text"]:
        out[col] = out[col].map(_apply)

    return out

def add_decision_text_features(behav: pd.DataFrame, snt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds decision-level text features to behav:
      - opt1_text, opt2_text
      - chosen_text, unchosen_text
      - selected_option (0=opt1, 1=opt2; nullable Int64)
      - prevslide_text (previous slide's realized text; if prev was Decision => previous chosen_text)
      - num_total_words (+ a few small helpful word-count columns)
      - decision_prompt_text, prevslide_type (small debugging helpers)

    Requires:
      behav columns: ['decision_num','dimension'] and '{dimension}_decision' for dimension in {'affil','power'}.
      snt_df decision rows contain: ['decision_num','opt1_text','opt2_text','opt1_{dim}','opt2_{dim}'].
    """
    
    def _word_count(x) -> int:
        if not isinstance(x, str):
            return 0
        x = x.strip()
        return 0 if x == "" else len(x.split())

    out = behav.copy()

    # ---- ensure decision_num exists ----
    if "decision_num" not in out.columns:
        out["decision_num"] = np.arange(1, len(out) + 1)

    # normalize decision_num to numeric int-like
    out["decision_num"] = pd.to_numeric(out["decision_num"], errors="coerce")

    # ---- task decision rows ----
    snt_dec = snt_df[snt_df["slide_type"] == "Decision"].copy()
    snt_dec["decision_num"] = pd.to_numeric(snt_dec["decision_num"], errors="coerce")

    # mapping: decision_num -> snt decision row (Series)
    snt_dec = snt_dec.dropna(subset=["decision_num"])
    if snt_dec["decision_num"].duplicated().any():
        dupes = snt_dec.loc[snt_dec["decision_num"].duplicated(), "decision_num"].tolist()
        raise ValueError(f"Duplicate decision_num in snt_df decision slides: {dupes[:10]} ...")
    snt_dec_map = snt_dec.set_index("decision_num", drop=False)

    # mapping: decision_num -> slide index in snt_df (for prevslide lookup)
    snt_dec_pos = (
        snt_df.reset_index(drop=True)
              .reset_index(names="slide_idx")
              .query("slide_type == 'Decision'")[["decision_num", "slide_idx"]]
              .copy()
    )
    snt_dec_pos["decision_num"] = pd.to_numeric(snt_dec_pos["decision_num"], errors="coerce")
    snt_dec_pos = snt_dec_pos.dropna(subset=["decision_num"])
    decnum_to_slideidx = dict(zip(snt_dec_pos["decision_num"], snt_dec_pos["slide_idx"]))

    N = len(out)

    # preallocate
    opt1_text = [np.nan] * N
    opt2_text = [np.nan] * N
    decision_prompt_text = [np.nan] * N

    chosen_text = [np.nan] * N
    unchosen_text = [np.nan] * N
    selected_option = [pd.NA] * N  # nullable ints

    opt1_words = [0] * N
    opt2_words = [0] * N
    chosen_words = [pd.NA] * N
    unchosen_words = [pd.NA] * N
    num_total_words = [0] * N

    # ---- pass 1: per-decision chosen/unchosen ----
    for i in range(N):
        dnum = out["decision_num"].iloc[i]
        if not np.isfinite(dnum):
            continue
        dnum = int(dnum)

        if dnum not in snt_dec_map.index:
            # task file mismatch; leave NaNs (but keep processing)
            continue

        srow = snt_dec_map.loc[dnum]

        # store raw option texts (always useful)
        o1 = srow.get("opt1_text", np.nan)
        o2 = srow.get("opt2_text", np.nan)
        opt1_text[i] = o1
        opt2_text[i] = o2

        # prompt text (if present in the sheet)
        decision_prompt_text[i] = srow.get("text", np.nan)

        # word counts for options (always definable if texts exist)
        w1 = _word_count(o1)
        w2 = _word_count(o2)
        opt1_words[i] = w1
        opt2_words[i] = w2
        num_total_words[i] = w1 + w2

        dim = str(out["dimension"].iloc[i]).strip().lower()

        # Only affil/power decisions can be mapped by direction using opt1_{dim}, opt2_{dim}
        if dim in ("affil", "power"):
            choice_col = f"{dim}_decision"
            if choice_col not in out.columns:
                raise ValueError(f"Behavior file missing required column {choice_col!r} for dim={dim!r}.")

            choice_dir = pd.to_numeric(out[choice_col].iloc[i], errors="coerce")
            opt1_dir = pd.to_numeric(srow.get(f"opt1_{dim}", np.nan), errors="coerce")
            opt2_dir = pd.to_numeric(srow.get(f"opt2_{dim}", np.nan), errors="coerce")

            # if no response / cannot interpret, leave chosen/selected as NA
            if np.isfinite(choice_dir) and choice_dir != 0 and np.isfinite(opt1_dir) and np.isfinite(opt2_dir):
                if np.isclose(choice_dir, opt1_dir):
                    chosen_text[i] = o1
                    unchosen_text[i] = o2
                    selected_option[i] = 0
                elif np.isclose(choice_dir, opt2_dir):
                    chosen_text[i] = o2
                    unchosen_text[i] = o1
                    selected_option[i] = 1
                # else: unexpected coding; leave NA

        # For dim not in ('affil','power'), we keep opt1/opt2 but do not guess chosen.
        # (If you later confirm how neutral trials encode choice, you can extend here.)

        # chosen/unchosen word counts (only if chosen is known)
        if isinstance(chosen_text[i], str):
            chosen_words[i] = _word_count(chosen_text[i])
        if isinstance(unchosen_text[i], str):
            unchosen_words[i] = _word_count(unchosen_text[i])

    # build helper dict for prevslide decision lookup: decision_num -> chosen_text
    chosen_by_decnum = {}
    for i in range(N):
        dnum = out["decision_num"].iloc[i]
        if np.isfinite(dnum):
            chosen_by_decnum[int(dnum)] = chosen_text[i]

    # ---- pass 2: prevslide_text ----
    prevslide_text = [np.nan] * N
    prevslide_type = [np.nan] * N

    for i in range(N):
        dnum = out["decision_num"].iloc[i]
        if not np.isfinite(dnum):
            continue
        dnum = int(dnum)

        slide_idx = decnum_to_slideidx.get(dnum, None)
        if slide_idx is None or slide_idx <= 0:
            continue

        prev_row = snt_df.iloc[int(slide_idx) - 1]
        ptype = prev_row.get("slide_type", np.nan)
        prevslide_type[i] = ptype

        if ptype == "Decision":
            prev_dnum = pd.to_numeric(prev_row.get("decision_num", np.nan), errors="coerce")
            if np.isfinite(prev_dnum):
                prevslide_text[i] = chosen_by_decnum.get(int(prev_dnum), np.nan)
        else:
            prevslide_text[i] = prev_row.get("text", np.nan)

    # ---- assign to dataframe ----
    out["opt1_text"] = opt1_text
    out["opt2_text"] = opt2_text
    out["decision_prompt_text"] = decision_prompt_text

    out["chosen_text"] = chosen_text
    out["unchosen_text"] = unchosen_text
    out["selected_option"] = pd.Series(selected_option, index=out.index, dtype="Int64")

    out["prevslide_text"] = prevslide_text
    out["prevslide_type"] = prevslide_type

    out["opt1_words"] = opt1_words
    out["opt2_words"] = opt2_words
    out["num_total_words"] = num_total_words
    out["chosen_words"] = pd.Series(chosen_words, index=out.index, dtype="Int64")
    out["unchosen_words"] = pd.Series(unchosen_words, index=out.index, dtype="Int64")

    return out

# snt info
snt_df = pd.read_excel('../data/info/social-navigation-task.xlsx')
snt_df = snt_df[np.isfinite(snt_df['trial_num'])]
snt_df = snt_df.sort_values(by='trial_num')
snt_df = snt_df[snt_df['slide_type'] != 'Game over']
snt_df = snt_df[snt_df['slide_type'] != 'Image']
snt_df.reset_index(drop=True, inplace=True)
snt_df['word_count'] = snt_df['text'].apply(lambda x: len(x.split())) # word count 

genders = ['woman', 'man', 'man', 'man', 'woman', 'woman']
character_genders = {'First': genders[0], 'Second': genders[1], 'Assistant': genders[2],
                    'Newcomb':genders[3], 'Hayworth': genders[4], 'Neutral': genders[5]}
snt_gendered_df = replace_pronouns(snt_df, character_genders) # just choosing an arbitrary version right now

#-------------------- social space features


#-------------------- run it per subject
# other-samples/tavares/
behav_fnames = glob.glob("../data/other-samples/tavares/preprocessed/behavior/*.xlsx")
print(f"Found {len(behav_fnames)} behavioral files")

for behav_fname in behav_fnames:

    print(f"Processing {behav_fname}...")
    behav = pd.read_excel(behav_fname)
    dim   = behav["dimension"].astype(str).str.lower()

    # ------------------------------------------------------------
    # locations: past, present, counterfactual
    # ------------------------------------------------------------

    #---------- convert choices to coordinates
    choices = behav[['affil_decision', 'power_decision']].to_numpy() # (N,2)
    coords  = apply_by_character(choices, compute_coords_from_choices) # (N,2)
    if ("affil_coord" not in behav.columns) or ("power_coord" not in behav.columns):
        behav["affil_coord"] = coords[:, 0]
        behav["power_coord"] = coords[:, 1]

    #---------- Previous location per character (shift within character)
    prev = (behav.groupby("character_role_num", sort=False)[["affil_coord", "power_coord"]].shift(1))
    behav["affil_coord_prev"] = prev["affil_coord"].fillna(0.0)
    behav["power_coord_prev"] = prev["power_coord"].fillna(0.0)

    #---------- Counterfactual location if they made the OTHER choice
    responded = pd.to_numeric(behav["responded"], errors="coerce").fillna(0).astype(int).astype(bool)
    aff_dec   = pd.to_numeric(behav["affil_decision"], errors="coerce")
    pow_dec   = pd.to_numeric(behav["power_decision"], errors="coerce")

    # initialize as NaN everywhere
    behav["affil_coord_counterfactual"] = np.nan
    behav["power_coord_counterfactual"] = np.nan
    valid_dim = dim.isin(["affil", "power"])
    ok = responded & valid_dim

    # start from previous location & apply the *flipped* step on the active dimension
    behav.loc[ok, "affil_coord_counterfactual"] = behav.loc[ok, "affil_coord_prev"]
    behav.loc[ok, "power_coord_counterfactual"] = behav.loc[ok, "power_coord_prev"]
    m_aff = responded & (dim == "affil") & np.isfinite(aff_dec)
    behav.loc[m_aff, "affil_coord_counterfactual"] = behav.loc[m_aff, "affil_coord_prev"] - aff_dec.loc[m_aff]
    m_pow = responded & (dim == "power") & np.isfinite(pow_dec)
    behav.loc[m_pow, "power_coord_counterfactual"] = behav.loc[m_pow, "power_coord_prev"] - pow_dec.loc[m_pow]

    # Verify that "chosen" coords match: prev + decision on the active dimension (responded trials)
    tol = 1e-6
    bad_rows = []
    m_aff_chk = responded & (dim == "affil") & np.isfinite(aff_dec)
    if m_aff_chk.any():
        exp_aff = behav.loc[m_aff_chk, "affil_coord_prev"] + aff_dec.loc[m_aff_chk]
        exp_pow = behav.loc[m_aff_chk, "power_coord_prev"]
        bad = (
            ~np.isclose(behav.loc[m_aff_chk, "affil_coord"], exp_aff, atol=tol, rtol=0) |
            ~np.isclose(behav.loc[m_aff_chk, "power_coord"], exp_pow, atol=tol, rtol=0)
        )
        if bad.any():
            bad_rows.extend(behav.index[m_aff_chk][bad].to_list())
    m_pow_chk = responded & (dim == "power") & np.isfinite(pow_dec)
    if m_pow_chk.any():
        exp_pow = behav.loc[m_pow_chk, "power_coord_prev"] + pow_dec.loc[m_pow_chk]
        exp_aff = behav.loc[m_pow_chk, "affil_coord_prev"]
        bad = (
            ~np.isclose(behav.loc[m_pow_chk, "power_coord"], exp_pow, atol=tol, rtol=0) |
            ~np.isclose(behav.loc[m_pow_chk, "affil_coord"], exp_aff, atol=tol, rtol=0)
        )
        if bad.any():
            bad_rows.extend(behav.index[m_pow_chk][bad].to_list())
    if bad_rows:
        print(f"WARNING: {behav_fname} coordinate QC mismatch in {len(bad_rows)} trials. Example rows: {bad_rows[:10]}")

    # ------------------------------------------------------------
    # other behavioral features based on choices
    # ------------------------------------------------------------


    # affiliation and power
    runmean = apply_by_character(choices, compute_running_mean_from_choices) # (N,2)

    # POV angle and distance (tavares 2015)
    polar_pov = apply_by_character(choices, compute_tavares_from_choices, mode="pov")

    # consistency-like
    consistency = apply_by_character(choices, compute_consistency_from_choices, keep_all_trials=False)  # (N,1)

    behav['affil_runmean'] = runmean[:, 0]
    behav['power_runmean'] = runmean[:, 1]
    behav["distance"]      = polar_pov[:, 0]
    behav["theta"]         = polar_pov[:, 1]
    behav['consistency']   = consistency

    # ------------------------------------------------------------
    # NLP features
    # ------------------------------------------------------------

    behav = add_decision_text_features(behav, snt_gendered_df)

    behav.to_excel(behav_fname, index=False)
    print(f"  Saved updated behavioral file to {behav_fname}.")


Found 21 behavioral files
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-19.xlsx...
  Saved updated behavioral file to ../data/other-samples/tavares/preprocessed/behavior/sub-19.xlsx.
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-03.xlsx...
  Saved updated behavioral file to ../data/other-samples/tavares/preprocessed/behavior/sub-03.xlsx.
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-15.xlsx...
  Saved updated behavioral file to ../data/other-samples/tavares/preprocessed/behavior/sub-15.xlsx.
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-14.xlsx...
  Saved updated behavioral file to ../data/other-samples/tavares/preprocessed/behavior/sub-14.xlsx.
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-02.xlsx...
  Saved updated behavioral file to ../data/other-samples/tavares/preprocessed/behavior/sub-02.xlsx.
Processing ../data/other-samples/tavares/preprocessed/behavior/sub-18.xlsx...
  Save

## visualize RDMs

In [ ]:
# loop over ALL subjects in the folder....
for sub_id in incl_subs:

    #------------------- organize data

    behav  = load_behavior(sub_id)

    # convert choices to coordinates
    choices = behav[['affil_decision', 'power_decision']].to_numpy() # (N,2)
    coords = apply_by_character(choices, compute_coords_from_choices) # (N,2)
    coords_by_char = organize_by_character(coords)

    # affiliation and power
    runmean = apply_by_character(choices, compute_running_mean_from_choices) # (N,2)
    affil_mean, power_mean = coords.mean(axis=0) # overall mean
    affil_means, power_means = np.array([c.mean(axis=0) for c in coords_by_char]).T # per-character means

    # consistency
    consistency = apply_by_character(choices, compute_consistency_from_choices, keep_all_trials=False)  # (N,1)
    consistency_by_char = organize_by_character(consistency) # list of arrays (n_char_i, 1)
    consistency_char = np.array([np.nanmean(c[:, 0]) for c in consistency_by_char]) # average consistency per character
    consistency_avg = np.nanmean(consistency_char) # average consistency across characters 

    # perimeter and area
    cvhull = ConvexHull(coords) 
    perimeter, area = np.array([cvhull.area, cvhull.volume])

    # add variables to behavior
    behav['affil_runmean'] = runmean[:, 0]
    behav['power_runmean'] = runmean[:, 1]
    behav['consistency']   = consistency

    #------------------- plot

    fig, axs = plt.subplots(1, 9, figsize=(22, 5))

    ax = axs[0]
    ax.set_title("Onset")
    plot_rdm(behav['onset'].values[:,None], metric='euclidean', ax=ax);

    ax = axs[1]
    ax.set_title("Scene")
    plot_rdm(behav['scene_num'].values[:,None], metric='hamming', ax=ax);

    ax = axs[2]
    ax.set_title("Character ID")
    plot_rdm(behav['character_role_num'].values[:,None], metric='hamming', ax=ax);

    ax = axs[3]
    ax.set_title("Dimension")
    plot_rdm(behav['dimension'].values[:,None], metric='hamming', ax=ax);

    ax = axs[4]
    ax.set_title("Choices")
    plot_rdm(choices, metric='cosine', ax=ax);

    ax = axs[5]
    ax.set_title("Reaction time")
    plot_rdm(behav['reaction_time'].values[:,None], metric='euclidean', ax=ax);

    ax = axs[6]
    ax.set_title("Coordinates")
    plot_rdm(coords, metric='euclidean', ax=ax);

    ax = axs[7]
    ax.set_title("Running Mean")
    plot_rdm(runmean, metric='euclidean', ax=ax);

    ax = axs[8]
    ax.set_title("Consistency")
    plot_rdm(consistency, metric='euclidean', ax=ax);

    for ax in axs:
        ax.set_xlabel("Trial")
        ax.set_ylabel("Trial")
        ax.get_figure().axes[-1].remove()